# Setup

In [1]:
%load_ext autoreload
%autoreload 2

## IMPORTS
import os

import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from shapely.geometry import Point

import src.data_filepaths as data_filepaths
import src.utils.loaders as loaders
import src.utils.vis.colors_utils as colors_utils
from src.utils.df_tools import map_values_b_to_bp
from src.utils.loaders import process_loaded_building_df
from src.utils.vis.map import CategoricalColorMap, MapExplorer

# configs
sns.set_theme()

## Previous Steps

Pre-processing

This takes about 25s

In [2]:
b_gdf = process_loaded_building_df(gpd.read_parquet(data_filepaths.OVERTURE_BUILDING))
bp_gdf = process_loaded_building_df(
    gpd.read_parquet(data_filepaths.OVERTURE_BUILDING_PART)
)

osm_landuse_gdf = loaders.load_osm_landuse_1_geojson_to_gdf(data_filepaths.OSM_landuse)
mp_landuse_gdf = loaders.load_masterplan_landuse_geojson_to_gdf(
    data_filepaths.MASTERPLAN_LANDUSE
)

In [3]:
def merge_with_landuse(
    b_gdf: gpd.GeoDataFrame,
    bp_gdf: gpd.GeoDataFrame,
):
    buildings_osm_merge = gpd.sjoin(
        b_gdf[["centroid"]].set_geometry("centroid"),
        osm_landuse_gdf,
        how="left",
        predicate="intersects",
    )
    # deduplicate
    buildings_osm_merge = buildings_osm_merge[
        ~buildings_osm_merge.index.duplicated(keep="first")
    ]

    buildings_osm_merge = buildings_osm_merge.drop(columns="index_right")

    buildings_mp_merge = gpd.sjoin(
        b_gdf[["centroid"]].set_geometry("centroid"),
        mp_landuse_gdf,
        how="left",
        predicate="intersects",
    )
    # deduplicate
    buildings_mp_merge = buildings_mp_merge[
        ~buildings_mp_merge.index.duplicated(keep="first")
    ]

    buildings_mp_merge["landuse"].value_counts(dropna=False)

    # no assigning to b_gdf is being done
    return buildings_osm_merge, buildings_mp_merge


def preprocess_remove_buildings(
    b_gdf: gpd.GeoDataFrame,
    bp_gdf: gpd.GeoDataFrame,
    buildings_osm_merge,  # [landuse]
    buildings_mp_merge,  # [landuse, gpr]
    mp_landuse_gdf,
):
    """
    Adds removal reasons to b_gdf, and maps to bp_gdf as well.

    Creates new columns `removal_reason`, `is_removed`
    """

    SIZE_THRESHOLDS = {
        "OSM": 70,  # this number is quite high, but for demand I think it's fine
        "Google": 400,
        "Microsoft": 400,
    }
    MP_COVERAGE_THRESHOLD = 0.05
    b_gdf = b_gdf.copy()
    bp_gdf = bp_gdf.copy()

    # clean removals that have no other condition
    mp_landuses_remove = [
        "HEALTH & MEDICAL CARE",
        "EDUCATIONAL INSTITUTION",
        "PLACE OF WORSHIP",
        "CIVIC & COMMUNITY INSTITUTION",
        # "OPEN SPACE",
        "PARK",
        "BEACH AREA",
        "SPORTS & RECREATION",
        "WATERBODY",
        "ROAD",
        "TRANSPORT FACILITIES",
        "MASS RAPID TRANSIT",
        "LIGHT RAPID TRANSIT",
        "UTILITY",
        "CEMETERY",
        "PORT / AIRPORT",
        # "RESERVE SITE",
        "SPECIAL USE",
    ]
    MP_LANDUSE_REMOVAL_DICT = {landuse: "Landuse-MP" for landuse in mp_landuses_remove}

    b_gdf[["mp_landuse"]] = buildings_mp_merge[["landuse"]]
    b_gdf[["mp_gpr"]] = buildings_mp_merge[["gpr"]]
    b_gdf[["mp_OBJECTID"]] = buildings_mp_merge[["OBJECTID"]]

    # first calculate areas of a zone
    # this makes an assumption that building footprints are fully inside their zone
    # which is not true. but I can't be bothered
    b_mp_covered = b_gdf.groupby("mp_OBJECTID")["area_m2"].sum()
    mp_area = mp_landuse_gdf.geometry.area

    b_mp_coverage_prop = b_gdf["mp_OBJECTID"].map(b_mp_covered / mp_area)

    ###################################

    # now asasign all reasons at once
    b_gdf["removal_reason"] = np.select(
        [
            (buildings_osm_merge["landuse"] == "construction")
            | (buildings_osm_merge["landuse"] == "industrial_port"),
            (b_gdf["src_dataset_label"] == "OpenStreetMap")
            & (b_gdf["area_m2"] < SIZE_THRESHOLDS["OSM"]),
            (b_gdf["src_dataset_label"] == "Google Open Buildings")
            & (b_gdf["area_m2"] < SIZE_THRESHOLDS["Google"]),
            (b_gdf["src_dataset_label"] == "Microsoft ML Buildings")
            & (b_gdf["area_m2"] < SIZE_THRESHOLDS["Microsoft"]),
            buildings_mp_merge["landuse"].isna(),  # remove NaNs here, about 130 of them
            b_mp_coverage_prop < MP_COVERAGE_THRESHOLD,
        ],
        [
            "Landuse-OSM",
            "Size-OSM",
            "Size-Google",
            "Size-Microsoft",
            "Landuse-MP",
            "Landuse-MP_coverage",
        ],
        default=None,
    )

    # apply without overwriting non-matches
    b_gdf["removal_reason"] = (
        buildings_mp_merge["landuse"]
        .map(MP_LANDUSE_REMOVAL_DICT)
        .fillna(b_gdf["removal_reason"])
    )

    map_values_b_to_bp(b_gdf, bp_gdf, "removal_reason", "removal_reason")

    print(b_gdf["removal_reason"].value_counts(dropna=False))

    # create boolean removed columns
    b_gdf["is_removed"] = ~b_gdf["removal_reason"].isna()
    bp_gdf["is_removed"] = ~bp_gdf["removal_reason"].isna()

    return b_gdf, bp_gdf

In [4]:
buildings_osm_merge, buildings_mp_merge = merge_with_landuse(b_gdf, bp_gdf)

b_gdf1, bp_gdf1 = preprocess_remove_buildings(
    b_gdf,
    bp_gdf,
    buildings_osm_merge,
    buildings_mp_merge,
    mp_landuse_gdf,
)

removal_reason
NaN                    108252
Size-Google             56897
Landuse-MP              41003
Size-Microsoft          19362
Size-OSM                 5748
Landuse-OSM              4742
Landuse-MP_coverage      1829
Name: count, dtype: int64


In [5]:
b_gdf1.head(3)

,names,sources,height,min_height,is_underground,num_floors,num_floors_underground,min_floor,geometry,subtype,class,has_parts,src_dataset_label,centroid,area_m2,mp_landuse,mp_gpr,mp_OBJECTID,removal_reason,is_removed
id,,,,,,,,,,,,,,,,,,,,
e71fc534-b314-449f-9ec9-1e93ab103733,None,"[{'property': '', 'dataset': 'Microsoft ML Bui...",NaN,NaN,False,NaN,NaN,NaN,"POLYGON ((5872.709 22340.088, 5882.848 22339.7...",NaN,NaN,False,Microsoft ML Buildings,POINT (5877.918 22344.533),93.248417,PORT / AIRPORT,0.0,619836.0,Landuse-MP,True
af894262-c019-4f4d-9b7c-71c934d25f9e,None,"[{'property': '', 'dataset': 'Google Open Buil...",NaN,NaN,False,NaN,NaN,NaN,"POLYGON ((6072.91 21627.47, 6076.606 21639.501...",NaN,NaN,False,Google Open Buildings,POINT (6069.226 21635.181),145.675822,RESERVE SITE,0.0,619284.0,Landuse-OSM,True
4a403634-2a1c-470d-8272-ed1283c02a6c,None,"[{'property': '', 'dataset': 'Google Open Buil...",NaN,NaN,False,NaN,NaN,NaN,"POLYGON ((6079.655 21624.617, 6083.183 21628.7...",NaN,NaN,False,Google Open Buildings,POINT (6079.727 21628.12),24.139091,RESERVE SITE,0.0,619284.0,Landuse-OSM,True


## Re-add Specific Buildings

see removal visualization from before

In [6]:
def re_add_buildings(b_gdf, bp_gdf, buildings_mp_merge):
    """
    Assigns some specified buildings to be readded.
    `removal_reason` is reset to `NaN`, and these buildings are given a new
    MP plot zoning binding in `buildings_mp_merge` with `landuse=READDED`.
    This is so that they can be included with the processing for buildings with `landuse`
    `OPEN SPACE`, `RESERVE SITE`, etc.

    Returns modified `b_gdf`, `bp_gdf` (with removal reason mapped) and `buildings_mp_merge`.
    """
    b_gdf = b_gdf.copy()
    bp_gdf = bp_gdf.copy()
    buildings_mp_merge = buildings_mp_merge.copy()

    ref_pts = gpd.GeoDataFrame(
        geometry=[
            Point(103.989745, 1.360133),  # Jewel
            Point(103.790116, 1.275532),  # PSA Horizons
            Point(103.816265, 1.279862),  # DSTA Tower A
            Point(103.764051, 1.366826),  # CMPB
            Point(103.848157, 1.285193),  # Pickering Operation Complex
            # SIA Airline House; currently not office but will be in next update
            Point(103.996845, 1.377423),
            # SIA Training Centre; currently not office but will be in next update
            Point(103.965935, 1.345763),
            # changi airport cargo areas, most currently not correctly tagged, but will be in future
            Point(103.993964, 1.374522),  # 1-4
            Point(103.995509, 1.378244),  # 5
            Point(103.996625, 1.380893),  # 6
            Point(103.996260, 1.372398),  # dnata
        ],
        crs="EPSG:4326",  # longitude/latitude
    ).to_crs(
        "EPSG:3414"
    )  # singapore CRS

    matches = gpd.sjoin(
        b_gdf,
        ref_pts,
        how="inner",
        predicate="contains",
    )

    b_gdf.loc[matches.index, "is_removed"] = False
    b_gdf.loc[matches.index, "mp_landuse"] = "READDED"
    buildings_mp_merge.loc[matches.index, "landuse"] = "READDED"

    # print(buildings_mp_merge["landuse"].value_counts())

    map_values_b_to_bp(b_gdf, bp_gdf, "is_removed", "is_removed")

    return b_gdf, bp_gdf, buildings_mp_merge


# make sure to verify
b_gdf1, bp_gdf1, buildings_mp_merge = re_add_buildings(
    b_gdf1, bp_gdf1, buildings_mp_merge
)

## Remove

Remove those with `is_removed == False`

In [7]:
b_gdf_rmv = b_gdf1[~b_gdf1["is_removed"]].drop(columns=["is_removed", "removal_reason"])
bp_gdf_rmv = bp_gdf1[~bp_gdf1["is_removed"]].drop(
    columns=["is_removed", "removal_reason"]
)

print(
    f"{len(b_gdf1) - len(b_gdf_rmv)}/{len(b_gdf1)} buildings removed: "
    + f"{((len(b_gdf1) - len(b_gdf_rmv)) / len(b_gdf1) * 100):.3f}%"
)
print(
    f"{len(bp_gdf1) - len(bp_gdf_rmv)}/{len(bp_gdf1)} building parts removed: "
    + f"{((len(bp_gdf1) - len(bp_gdf_rmv)) / len(bp_gdf1) * 100):.3f}%"
)

129569/237833 buildings removed: 54.479%
1636/8173 building parts removed: 20.017%


# Building Demand Type Classification

In [8]:
from enum import Enum

class Zoning(Enum):
    RES = "RESIDENTIAL"
    RES1C = "RESIDENTIAL WITH COMMERCIAL AT 1ST STOREY"
    RESCOM = "COMMERCIAL & RESIDENTIAL"
    COMRES = "COMMERCIAL & RESIDENTIAL" # alternative
    COM = "COMMERCIAL"
    HOTEL = "HOTEL"
    WHITE = "WHITE"
    BP = "BUSINESS PARK"
    BPW = "BUSINESS PARK - WHITE"
    B1 = "BUSINESS 1"
    B2 = "BUSINESS 2"
    B1W = "BUSINESS 1 - WHITE"
    B2W = "BUSINESS 2 - WHITE"
    RESINST = "RESIDENTIAL / INSTITUTION"
    COMINST = "COMMERCIAL / INSTITUTION"
    AGRI = "AGRICULTURE"

## A - Merge Building Parts

- A: Merge building parts into buildings
    - for every building with parts, IF the building doesn’t have `num_floors` already:
        - note: bparts don’t have `class` and `subtype`
        - account for `min_floor`
        - merge into building based on building’s footprint, assign `num_floors`

In [9]:
def temp_vis_A():
    """
    Visualizes buildings that have parts and num_floors
    """
    temp_b_gdf = b_gdf_rmv.copy()
    temp_bp_gdf = bp_gdf_rmv.copy()

    temp_b_gdf["has_parts_and_num_floors"] = (
        temp_b_gdf["has_parts"] & ~temp_b_gdf["num_floors"].isna()
    )

    print(temp_b_gdf["has_parts_and_num_floors"].sum()) # 323

    map_values_b_to_bp(
        temp_b_gdf, temp_bp_gdf, "has_parts_and_num_floors", "has_parts_and_num_floors"
    )
    map_values_b_to_bp(temp_b_gdf, temp_bp_gdf, "has_parts", "has_parts")

    # cmap = CategoricalColorMap(
    #     cat_colors=list(mp_lu_color_mapping.values()),
    #     cat_labels=list(mp_lu_color_mapping.keys()),
    #     nan_color=np.array([0.2, 0.2, 0.2, 0.5]),
    # )

    mapper = MapExplorer(
        temp_b_gdf,
        temp_bp_gdf,
        # enable_debug_mode=True,
    )
    return mapper.build_singleprop(
        "has_parts_and_num_floors",
        # cmap,
        title="Buildings that have parts and num_floors",
    )

# temp_vis_A()

THERE ARE STILL EXISTING PROBLEMS
* Expo

In [10]:
def merge_building_parts_into_buildings(
    b_gdf: gpd.GeoDataFrame, bp_gdf: gpd.GeoDataFrame
):
    """
    Adds the floorspace of building parts into buildings, so building parts do not need to be
    considered anymore for demand calculation

    `b_gdf`, `bp_gdf` should be the overall GDFs (the result after prior building removals),
    no specific selection of buildings with associated
    parts should be made.

    This will them modify existing buildings' `num_levels` based on the summation of floorspace
    of its constituent parts. Only parts with `num_levels` and `min_level` are to be counted,
    as building parts that are specified **only** with rendering information (height, etc.)
    are just visual parts used to construct 3D buildings.

    For buildings that have existing `num_levels` (around 300 of them), this can point to two reasons:
    * (The correct OSM way) - the building's floor count refers to the max/min floors of a building.
        This is quite rare in Singapore, and is easily filtered out by checking max floors of a building
        against its parts. If the maximums are equal, then this building's `num_levels` will be ignored
        and taken entirely from its parts.
    * The more common tagging method in Singapore is for the building to represent some kind of lower-level
        structure, such as a carpark, or commercial building, underneath apartment tower blocks.
        To account for this, building parts will add onto the base building's num_levels.
        Of course, there are yet more asterisks because *most* building_parts don't get correctly tagged with
        minimum floors and as such vertically overlap. But I will not handle that. A notable example where
        this problem does crop up is *Suntec Tower 1*.
    """

    bp_gdf = bp_gdf.copy()
    # select buildings with parts
    b_gdf = b_gdf.copy()
    b_gdf_withparts = b_gdf[b_gdf.index.isin(bp_gdf["building_id"])][
        ["num_floors", "min_floor", "area_m2", "has_parts"]
    ]
    assert b_gdf_withparts[
        "has_parts"
    ].all(), "There are buildings in b_gdf with has_parts=False"

    print(len(b_gdf_withparts) - b_gdf_withparts["num_floors"].isna().sum())

    # allow buildings with no floors set (but potentially its constituent parts do) do accept their parts' floors
    b_gdf_withparts["num_floors"] = b_gdf_withparts["num_floors"].fillna(0)

    # if a building has at least the same num floors as the max of its parts, reset it to zero
    parts_max_floors = bp_gdf.groupby("building_id")["num_floors"].max()
    b_gdf_withparts.loc[
        b_gdf_withparts["num_floors"].ge(b_gdf_withparts.index.map(parts_max_floors)),
        "num_floors",
    ] = 0

    # add floor counts to bparts without floors but with height
    # set min_height as min_floor as well
    mask = bp_gdf["num_floors"].isna() & ~bp_gdf["height"].isna()
    bp_gdf.loc[mask, "num_floors"] = (bp_gdf.loc[mask, "height"]) / 4
    bp_gdf.loc[mask, "min_floor"] = (bp_gdf.loc[mask, "min_height"]) / 4

    def apply_fn(group_df: pd.DataFrame):
        # input - entire group's df
        # num_floors is guaranteed to be nonzero based on an earlier check
        x = group_df

        # DEBUGGING
        def debug():
            if x["building_id"].isin(["dd0993d9-306e-4cf0-8138-8d10df859d69"]).any():
                print("Parent building id:" + x.iloc[0]["building_id"])
                print(x[["area_m2", "num_floors", "min_floor"]])

        # debug()

        x["min_floor"] = x["min_floor"].fillna(0)
        # not even gonna check, if there are tagging errors, just assign it to zero
        x["net_floors"] = (x["num_floors"] - x["min_floor"]).clip(lower=0)

        floors_times_area = np.sum(x["net_floors"] * x["area_m2"])
        return floors_times_area

    agg = (
        bp_gdf[~bp_gdf["num_floors"].isna()]
        .groupby("building_id")[["area_m2", "num_floors", "min_floor", "building_id"]]
        .apply(apply_fn)
    )

    b_gdf_withparts["extra_floors_times_area"] = b_gdf_withparts.index.map(agg).fillna(
        0
    )

    # use .update() rather than direct assignment to not override existing num_floors
    b_gdf.update(
        # this is apparently necessary beacuse it's a copy
        {
            "num_floors": b_gdf_withparts["num_floors"]
            + b_gdf_withparts["extra_floors_times_area"] / b_gdf_withparts["area_m2"]
        }
    )

    # return b_gdf_withparts
    return b_gdf


# print(bp_gdf_rmv["min_height"])

b_gdf2 = merge_building_parts_into_buildings(b_gdf_rmv, bp_gdf_rmv)
print("\n### TOP 5 FLOORS:")
print(
    b_gdf2["num_floors"].sort_values(ascending=False).head(5)
)  # should have decimal places
# res[res["extra_floors_times_area"] > 0]


# MapExplorer(b_gdf2).build_singleprop(
#     "height_calc",
#     title="Buildings with new, combined heights",
#     continuous_cmap_name="YlGnBu",
# )

322

### TOP 5 FLOORS:
id
e8090893-44da-4ccd-b08f-625b7e2e6051    102.150457
19734297-5359-4084-9ee4-d60ceef87e2d     90.984333
0f81e0db-7122-4b99-9176-206a3297a95c     70.000000
43d2ac74-4c37-48af-bba3-89eb0e561786     69.222215
feca34e6-af9f-4003-a129-8b21339acc66     69.065354
Name: num_floors, dtype: float64


In [11]:
# MapExplorer(b_gdf2).build_singleprop(
#     "height_calc",
#     title="Buildings with new, combined heights",
#     continuous_cmap_name="YlGnBu",
# )

## Intermediate: Create new demand-type-building DF | And other setup

And generally cleaning up the dataframe for demand type classification & assignment

In [16]:
# include colormaps for demand types
DEMAND_TYPE_COLORMAP = {
    "res_low": "#98f6b0",
    "res_med": "#019529",
    "res_high": "#05696b",
    "office": "#ed0dd9",
    "commercial": "#5539cc",
    "ind_low": "#fffe7a",
    "ind_med": "#feb308",
    "ind_high": "#ff7855",
}
DEMAND_TYPE_CMAP_OBJ = CategoricalColorMap(
    cat_colors=list(DEMAND_TYPE_COLORMAP.values()),
    cat_labels=list(DEMAND_TYPE_COLORMAP.keys()),
    nan_color=np.array([0.8, 0.8, 0.8, 0.4])
)


from src.utils.vis.map_special import build_map_building_demand_type

In [17]:
# add all MP zonal information to b_gdf
b_gdf_withmp = b_gdf2
b_gdf_withmp[
    ["mp_white_q_max", "mp_gpr_b_min", "mp_is_landed", "mp_is_sdp"]
] = buildings_mp_merge[["white_q_max", "gpr_b_min", "is_landed", "is_sdp"]]

# further streamlining of dataset
b_gdf_withmp = b_gdf_withmp.drop(
    columns=[
        "sources",
        "src_dataset_label",
        "is_underground",
        "min_height",
        "has_parts",
        "names",
    ]
)
b_gdf_withmp["net_floors"] = (
    b_gdf_withmp["num_floors_underground"].fillna(0)
    + b_gdf_withmp["num_floors"]  # this one determines whether it's NaN or not
    - b_gdf_withmp["min_floor"].fillna(0)
)
b_gdf_withmp = b_gdf_withmp.drop(
    columns=["num_floors", "num_floors_underground", "min_floor"]
)

In [18]:
# demandtype buildings gdf
b_dmdtype_gdf = gpd.GeoDataFrame(
    columns=[
        "mp_OBJECTID",
        "mp_landuse",
        "demand_type",
        "geometry",
        "centroid",
        # existing values from building
        "b_id",
        "area_m2",
        "net_floors",
        "height", # to reverse estimate floorcount
        # new adjustment parameters when splitting buildings
        "floorcount_adj",
        "floorspace_coeff",
    ]
)
# new index, not using any existing index

MP columns:

* `mp_OBJECTID`
* `mp_landuse`, `mp_gpr`
* `mp_white_q_max`, `mp_gpr_b_min`, `mp_is_landed`, `mp_is_sdp`

## B - Handle Ambiguous MP Plots

- `RESERVE SITE`, `OPEN SPACE`, `is_sdp=True`
    - e.g. open_space - a couple of the big buildings at marina south are worth investigating
- Also `READDED`
- select only specific OSM tags, immediately **assign to its demand type**,
ignore others

In [19]:
def handle_ambiguous_mp_plots(b_gdf_withmp, b_dmdtype_gdf):
    """
    Returns a `b_dmdtype_gdf` with new appended values

    Only the following Overture building `class` values will be converted to `demand_type`:
    * residential -> res_high
    * apartments -> res_high
    * bungalow -> res_low
    * terrace -> res_low
    * detached -> res_low
    * semidetached_house -> res_low
    * office -> office
    * government -> office
    * public -> commercial
    * commercial -> commercial
    * retail -> commercial
    * supermarket -> commercial
    * industrial -> ind_med
    * manufacture -> ind_med
    * warehouse -> ind_low

    The rest, including NaN (OSM `building=yes`) will be dropped.
    """

    b_selection = b_gdf_withmp[
        b_gdf_withmp["mp_landuse"].isin(["RESERVE SITE", "OPEN SPACE", "READDED"])
        | b_gdf_withmp["mp_is_sdp"]
    ]
    initial_count = len(b_selection)
    # about 1700 buildings

    b_selection["demand_type"] = b_selection["class"].map(
        {
            "residential": "res_high",
            "apartments": "res_high",
            "bungalow": "res_low",
            "terrace": "res_low",
            "detached": "res_low",
            "semidetached_house": "res_low",
            "office": "office",
            "government": "office",
            "public": "commercial",
            "commercial": "commercial",
            "retail": "commercial",
            "supermarket": "commercial",
            "industrial": "ind_med",
            "manufacture": "ind_med",
            "warehouse": "ind_low",
            "hangar": "ind_low",
        }
    )
    new_count = b_selection["demand_type"].count()
    # down to 400 selected buildings lmao
    print(
        f"Out of {initial_count} total ambiguous buildings, "
        + f"{new_count} buildings ({(new_count/initial_count*100):.2f}%) successfully tagged with demand_type."
    )

    b_final = b_selection
    # b_final = b_selection[~b_selection["demand_type"].isna()]

    to_append = gpd.GeoDataFrame(
        {
            "mp_OBJECTID": b_final["mp_OBJECTID"],
            "mp_landuse": b_final["mp_landuse"],
            "demand_type": b_final["demand_type"],
            "geometry": b_final["geometry"],
            "centroid": b_final["centroid"],
            "b_id": b_final.index,
            "area_m2": b_final["area_m2"],
            # TODO might wanna improve, i.e. based on demand type
            # currently takes from height, assumes avr floor height of 4 m
            # and if nothing is specified, gives it 1 floor
            "net_floors": b_final["net_floors"].fillna(
                (b_final["height"] / 4).fillna(1)
            ),
            "height": b_final["height"],
            "floorcount_adj": 0,
            "floorspace_coeff": 1,
        }
    )

    return pd.concat([b_dmdtype_gdf, to_append])


res = handle_ambiguous_mp_plots(b_gdf_withmp, b_dmdtype_gdf.head(0))
build_map_building_demand_type(res, DEMAND_TYPE_CMAP_OBJ)

Out of 3755 total ambiguous buildings, 681 buildings (18.14%) successfully tagged with demand_type.


## D - Handle main 